# 01 — Build Single Source of Truth (`accounts.csv`)

Baut `data/accounts.csv` (Long-Format: eine Zeile pro Account × Kanal) aus:
- `data/adresspaket_pk.xlsx` — Bundestagsabgeordnete (MdBs)
- `data/dboes.csv` — DBoeS (News, Organisationen, Journalisten, Politiker)
- `data/stiftungen.csv` — Politische Stiftungen

Angereichert mit `rank` aus `data/Facebook_ranked.csv` und `data/Instagram_ranked.csv`.

**Schema:** `name, channel, handle, url, category, label, rank`

In [1]:
import os
from urllib.parse import urlparse

import pandas as pd

PROJECT_ROOT = os.path.dirname(os.path.dirname(os.path.abspath(os.getcwd() + "/scripts")))
# Robust egal ob als Notebook in scripts/ oder Repo-Root ausgeführt
if os.path.basename(os.getcwd()) == "scripts":
    PROJECT_ROOT = os.path.dirname(os.getcwd())
else:
    PROJECT_ROOT = os.getcwd()
DATA_DIR = os.path.join(PROJECT_ROOT, "data")

ADRESSPAKET = os.path.join(DATA_DIR, "adresspaket_pk.xlsx")
DBOES = os.path.join(DATA_DIR, "dboes.csv")
STIFTUNGEN = os.path.join(DATA_DIR, "stiftungen.csv")
FB_RANK = os.path.join(DATA_DIR, "Facebook_ranked.csv")
IG_RANK = os.path.join(DATA_DIR, "Instagram_ranked.csv")
OUTPUT = os.path.join(DATA_DIR, "accounts.csv")


def extract_handle_from_url(url):
    if pd.isna(url) or not isinstance(url, str):
        return None
    url = url.strip().rstrip("/")
    if url in (".", "existiert nicht", "", "nan"):
        return None
    try:
        parsed = urlparse(url)
        path = parsed.path.strip("/")
        if path:
            handle = path.split("/")[-1].split("?")[0].split("#")[0]
            if handle and handle != ".":
                return handle
    except Exception:
        pass
    return None


PARTY_CSV_MAP = {
    "CDU": "CDU", "CSU": "CSU", "SPD": "SPD",
    "B90/GRÜNE": "Grüne", "FDP": "FDP",
    "AfD": "AfD", "AFD": "AfD",
    "LINKE": "Linke", "BSW": "BSW",
}

def map_party_csv(partei):
    if pd.isna(partei) or partei in (".", ""):
        return None
    return PARTY_CSV_MAP.get(str(partei).strip(), "Sonstige Parteien")


PARTY_EXCEL_MAP = {
    "CDU": "CDU", "CSU": "CSU", "SPD": "SPD",
    "Bündnis 90/Die Grünen": "Grüne",
    "DIE LINKE": "Linke", "AfD": "AfD",
}

def map_party_excel(partei):
    if pd.isna(partei):
        return None
    return PARTY_EXCEL_MAP.get(str(partei).strip(), "Sonstige Parteien")


def clean_stiftung_handle(h):
    if pd.isna(h):
        return None
    s = str(h).strip().lstrip("@")
    if not s or s.lower() in ("nicht bekannt", "."):
        return None
    return s

## 1. MdBs aus Adresspaket (Basis)
Kanäle: TWITTER, FACEBOOK, Instagram, TikTok, Youtube

In [2]:
xls = pd.read_excel(ADRESSPAKET)
print(f"Adresspaket: {len(xls)} Zeilen")

_excel_channel_cols = [
    ("TWITTER", "x"),
    ("FACEBOOK", "facebook"),
    ("Instagram", "instagram"),
    ("TikTok", "tiktok"),
    ("Youtube", "youtube"),
]

mdb_rows = []
for _, row in xls.iterrows():
    name = f"{row.get('VORNAME', '') or ''} {row.get('NACHNAME', '') or ''}".strip()
    label = map_party_excel(row.get("PARTEI"))
    for col, channel in _excel_channel_cols:
        url = row.get(col)
        handle = extract_handle_from_url(url)
        if not handle:
            continue
        mdb_rows.append({
            "name": name,
            "channel": channel,
            "handle": handle,
            "url": url,
            "category": "MdB",
            "label": label,
        })

df_mdb = pd.DataFrame(mdb_rows)
print(f"MdB-Zeilen: {len(df_mdb)}")
df_mdb.head()

Adresspaket: 630 Zeilen
MdB-Zeilen: 1785


,name,channel,handle,url,category,label
0,Sanae Abdi,x,_sanaeabdi,https://x.com/_sanaeabdi,MdB,SPD
1,Sanae Abdi,facebook,sanaeabdispd,https://m.facebook.com/sanaeabdispd/,MdB,SPD
2,Sanae Abdi,instagram,sanae_ccaa,https://www.instagram.com/sanae_ccaa/,MdB,SPD
3,Knut Abraham,x,Knut_Abraham,https://x.com/Knut_Abraham,MdB,CDU
4,Knut Abraham,facebook,abraham.knut,https://m.facebook.com/abraham.knut,MdB,CDU


## 2. dboes einlesen und splitten

In [3]:
dboes = pd.read_csv(DBOES)
print(f"dboes: {len(dboes)} Zeilen")
print("Kategorie-Verteilung:")
print(dboes["Kategorie"].value_counts().sort_index())

dboes: 7743 Zeilen
Kategorie-Verteilung:
Kategorie
1    2636
2     749
3    4358
Name: count, dtype: int64


## 3. dboes Kategorie 1 — News

In [4]:
TYP_NEWS_LABEL = {
    1: "Zeitung", 3: "Rundfunksender", 4: "Nachrichtenprogramm",
    5: "Entertainment", 6: "Online_Only", 7: "Nachrichtenagentur",
}

_dboes_channel_cols = [
    ("SM_XURL", "x"),
    ("SM_FacebookURL", "facebook"),
    ("SM_InstagramURL", "instagram"),
    ("SM_TikTokURL", "tiktok"),
]

def _clean_fb_id(v):
    """SM_FacebookID kann als float (100064181311439.0) reinkommen → als str ohne '.0'."""
    if pd.isna(v):
        return None
    s = str(v).strip()
    if s.endswith(".0"):
        s = s[:-2]
    return s or None


def dboes_long(sub_df, category_fn, label_fn):
    """Verwandelt eine dboes-Teilmenge in Long-Format.

    Setzt zusätzlich `_fb_id` auf Facebook-Zeilen (hilft beim Rank-Match),
    wird vor dem Speichern wieder entfernt.
    """
    out = []
    for _, row in sub_df.iterrows():
        cat = category_fn(row)
        lbl = label_fn(row)
        name = row.get("Name")
        name = "" if pd.isna(name) else str(name)
        for col, channel in _dboes_channel_cols:
            handle = extract_handle_from_url(row.get(col))
            if not handle:
                continue
            rec = {
                "name": name, "channel": channel,
                "handle": handle, "url": row.get(col),
                "category": cat, "label": lbl,
            }
            if channel == "facebook":
                rec["_fb_id"] = _clean_fb_id(row.get("SM_FacebookID"))
            out.append(rec)
    return pd.DataFrame(out)


df_news = dboes_long(
    dboes[dboes["Kategorie"] == 1],
    category_fn=lambda r: "News",
    label_fn=lambda r: TYP_NEWS_LABEL.get(int(r["Typ"])) if pd.notna(r["Typ"]) else None,
)
print(f"News (Kat 1): {len(df_news)} Zeilen")
df_news["label"].value_counts()

News (Kat 1): 4186 Zeilen


label
Zeitung                1697
Rundfunksender         1338
Online_Only             651
Entertainment           338
Nachrichtenprogramm     129
Nachrichtenagentur       33
Name: count, dtype: int64

## 4. dboes Kategorie 2 — Organisationen (Parteigliederung + Behörde)

In [5]:
def _org_label(row):
    typ = row.get("Typ")
    if pd.notna(typ) and int(typ) == 9:
        return map_party_csv(row.get("T_Partei"))
    if pd.notna(typ) and int(typ) == 15:
        return "Behörde"
    return None

df_org_dboes = dboes_long(
    dboes[dboes["Kategorie"] == 2],
    category_fn=lambda r: "Organisation",
    label_fn=_org_label,
)
print(f"Organisation (Kat 2): {len(df_org_dboes)} Zeilen")
df_org_dboes["label"].value_counts()

Organisation (Kat 2): 1866 Zeilen


label
Behörde              635
Sonstige Parteien    195
SPD                  181
Grüne                178
AfD                  170
CDU                  166
Linke                152
FDP                  151
BSW                   30
CSU                    8
Name: count, dtype: int64

## 5. dboes Kategorie 3 — Typ 20 (Journalisten) → News

In [6]:
df_journo = dboes_long(
    dboes[(dboes["Kategorie"] == 3) & (dboes["Typ"] == 20)],
    category_fn=lambda r: "News",
    label_fn=lambda r: "Journalist",
)
print(f"Journalisten (Kat 3 / Typ 20): {len(df_journo)} Zeilen")

Journalisten (Kat 3 / Typ 20): 1837 Zeilen


## 6. dboes Kategorie 3 — Typ 21 (Politiker) mit Match gegen Adresspaket
Wenn ein Handle (egal welcher Kanal) mit einem Adresspaket-Handle übereinstimmt,
überspringen wir den dboes-Eintrag (wird bereits als MdB geführt).
Sonst: `category="Politician"`, `label = T_Partei` gemappt.

In [7]:
mdb_handles_lower = set(df_mdb["handle"].dropna().str.lower())
print(f"MdB-Handles im Adresspaket: {len(mdb_handles_lower)}")

pol_rows_candidate = dboes[(dboes["Kategorie"] == 3) & (dboes["Typ"] == 21)]
print(f"dboes Typ 21 (Politiker): {len(pol_rows_candidate)} Einträge")

kept_rows = []
skipped = 0
for _, row in pol_rows_candidate.iterrows():
    raw_handles = []
    for col, _ch in _dboes_channel_cols:
        h = extract_handle_from_url(row.get(col))
        if h:
            raw_handles.append(h.lower())
    if any(h in mdb_handles_lower for h in raw_handles):
        skipped += 1
        continue
    kept_rows.append(row)

print(f"Übersprungen (Match mit Adresspaket): {skipped}")
print(f"Verbleibend als Politician: {len(kept_rows)}")

df_pol = dboes_long(
    pd.DataFrame(kept_rows),
    category_fn=lambda r: "Politician",
    label_fn=lambda r: map_party_csv(r.get("T_Partei")),
)
print(f"Politician-Zeilen: {len(df_pol)}")

MdB-Handles im Adresspaket: 1520
dboes Typ 21 (Politiker): 2807 Einträge


Übersprungen (Match mit Adresspaket): 492
Verbleibend als Politician: 2315


Politician-Zeilen: 6118


## 7. Stiftungen

In [8]:
stift = pd.read_csv(STIFTUNGEN)
print(f"Stiftungen: {len(stift)} Zeilen")

_stift_cols = [
    ("X_Twitter_Handle", "X_Twitter_URL", "x"),
    ("Facebook_Handle", "Facebook_URL", "facebook"),
    ("Instagram_Handle", "Instagram_URL", "instagram"),
]

stift_rows = []
for _, row in stift.iterrows():
    name = row.get("Stiftung", "")
    label = row.get("Partei")
    for h_col, u_col, channel in _stift_cols:
        handle = clean_stiftung_handle(row.get(h_col))
        if not handle:
            continue
        stift_rows.append({
            "name": name, "channel": channel,
            "handle": handle, "url": row.get(u_col),
            "category": "Organisation", "label": label,
        })

df_stift = pd.DataFrame(stift_rows)
print(f"Stiftungen-Zeilen: {len(df_stift)}")

Stiftungen: 13 Zeilen
Stiftungen-Zeilen: 36


## 8. Zusammenführen & Zeilen ohne Handle droppen

In [9]:
accounts = pd.concat(
    [df_mdb, df_news, df_org_dboes, df_journo, df_pol, df_stift],
    ignore_index=True,
)
print(f"Gesamt vor Cleanup: {len(accounts)}")
accounts = accounts[accounts["handle"].notna() & (accounts["handle"].astype(str).str.len() > 0)]
accounts = accounts.reset_index(drop=True)
print(f"Gesamt nach Cleanup: {len(accounts)}")

Gesamt vor Cleanup: 15828
Gesamt nach Cleanup: 15828


## 9. Rank-Matching (Facebook & Instagram)

In [10]:
fb_rank = pd.read_csv(FB_RANK)
ig_rank = pd.read_csv(IG_RANK)

# Instagram: Match ausschließlich per Handle (URL-Spalte existiert dort nicht).
ig_dict_handle = {
    str(h).lower(): int(r)
    for r, h in zip(ig_rank["Rank"], ig_rank["Account Handle"])
    if pd.notna(h)
}

# Facebook: zwei Lookups.
#   1) Per "Account Handle" (lowercase) — greift bei URL-Slug-Matches wie `spiegel.tv`.
#   2) Per numerischer FB-ID aus der URL-Spalte — greift bei dboes-Zeilen, die
#      SM_FacebookID tragen und somit mit der numerischen Rank-URL matchen.
fb_dict_handle = {
    str(h).lower(): int(r)
    for r, h in zip(fb_rank["Rank"], fb_rank["Account Handle"])
    if pd.notna(h)
}

def _fb_id_from_url(u):
    if pd.isna(u):
        return None
    last = str(u).rstrip("/").rsplit("/", 1)[-1]
    return last if last.isdigit() else None

fb_dict_id = {
    _fb_id_from_url(u): int(r)
    for r, u in zip(fb_rank["Rank"], fb_rank["URL"])
    if _fb_id_from_url(u) is not None
}

def lookup_rank(row):
    h = row["handle"]
    ch = row["channel"]
    if ch == "instagram":
        return ig_dict_handle.get(str(h).lower()) if pd.notna(h) else None
    if ch == "facebook":
        # Erst per FB-ID (nur bei dboes-Zeilen gesetzt), dann per Handle.
        fb_id = row.get("_fb_id")
        if fb_id and fb_id in fb_dict_id:
            return fb_dict_id[fb_id]
        if pd.notna(h):
            return fb_dict_handle.get(str(h).lower())
    return None

accounts["rank"] = accounts.apply(lookup_rank, axis=1)
accounts["rank"] = accounts["rank"].astype("Int64")

# Hilfsspalte wieder rauswerfen, gehört nicht ins finale Schema.
if "_fb_id" in accounts.columns:
    accounts = accounts.drop(columns=["_fb_id"])

print("Rank-Matches:")
fb_tot = (accounts["channel"] == "facebook").sum()
fb_hit = accounts[(accounts["channel"] == "facebook") & accounts["rank"].notna()].shape[0]
ig_tot = (accounts["channel"] == "instagram").sum()
ig_hit = accounts[(accounts["channel"] == "instagram") & accounts["rank"].notna()].shape[0]
print(f"  Facebook-Zeilen mit Rank: {fb_hit} / {fb_tot} ({fb_hit/fb_tot:.1%})")
print(f"  Instagram-Zeilen mit Rank: {ig_hit} / {ig_tot} ({ig_hit/ig_tot:.1%})")

Rank-Matches:
  Facebook-Zeilen mit Rank: 2506 / 4567 (54.9%)
  Instagram-Zeilen mit Rank: 3081 / 4837 (63.7%)


## 10. Sanity Checks

In [11]:
accounts = accounts[["name", "channel", "handle", "url", "category", "label", "rank"]]

print("== Kategorien ==")
print(accounts["category"].value_counts())
print()
print("== Channels ==")
print(accounts["channel"].value_counts())
print()
print("== Labels (Top 20) ==")
print(accounts["label"].value_counts(dropna=False).head(20))
print()
print("== Kombis category × channel ==")
print(accounts.groupby(["category", "channel"]).size().unstack(fill_value=0))

assert set(accounts["category"].unique()) <= {"MdB", "Politician", "News", "Organisation"}, "Unerwartete Kategorie!"
assert set(accounts["channel"].unique()) <= {"x", "facebook", "instagram", "tiktok", "youtube"}, "Unerwarteter Channel!"
assert accounts["handle"].notna().all() and (accounts["handle"].astype(str).str.len() > 0).all(), "Leerer Handle!"
print("\nAlle Asserts OK.")

== Kategorien ==
category
Politician      6118
News            6023
Organisation    1902
MdB             1785
Name: count, dtype: int64

== Channels ==
channel
instagram    4837
x            4742
facebook     4567
tiktok       1500
youtube       182
Name: count, dtype: int64

== Labels (Top 20) ==
label
SPD                                    2177
CDU                                    2042
Journalist                             1837
Zeitung                                1697
Grüne                                  1538
Rundfunksender                         1338
AfD                                    1279
Online_Only                             651
FDP                                     651
Behörde                                 635
Linke                                   581
Sonstige Parteien                       391
CSU                                     379
Entertainment                           338
Nachrichtenprogramm                     129
BSW                                

## 11. Speichern

In [12]:
accounts.to_csv(OUTPUT, index=False)
print(f"Gespeichert: {OUTPUT}")
print(f"Zeilen: {len(accounts)}")

Gespeichert: /Users/zorbeyozcan/Projekte/query_printer/data/accounts.csv
Zeilen: 15828
